# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [22]:
import os

# Define the path to the README file
readme_path = 'skills/README.md'

# Check if the file exists before trying to read it
if os.path.exists(readme_path):
    with open(readme_path, 'r') as f:
        readme_content = f.read()
    print(readme_content)
else:
    print(f"Error: The file '{readme_path}' was not found.")

# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame your question (ML-02, ML-03) | `framing-ml-problems/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Write +

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

For the initial feature vector, we will select `search_volume`, `competition`, `word_count`, and `char_count` from the `df` DataFrame. We'll also handle missing values in `word_count`, `char_count`, and `target_keyword_count` by filling them with 0 and creating new indicator columns to flag where values were originally missing.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [23]:
import numpy as np

# Initialize the feature vector X with selected features
X = df[['search_volume', 'competition', 'word_count', 'char_count']].copy()

# List of columns to flag missing values for
missing_cols_to_flag = ['word_count', 'char_count']

# Handle missing values: fill with 0 and create a '_is_missing' flag for specified columns
for col in missing_cols_to_flag:
    if col in X.columns:
        # Create a new column to indicate if the original value was missing
        X[f'{col}_is_missing'] = X[col].isnull().astype(int)
        # Fill missing values with 0
        X[col] = X[col].fillna(0)
    else:
        print(f"Warning: Column '{col}' not found in X. Skipping missing value handling for this column.")

# Display the first few rows of the engineered feature vector
display(X.head())

,search_volume,competition,word_count,char_count,word_count_is_missing,char_count_is_missing
0,10.0,0.67,3221.0,20457.0,0,0
1,90.0,0.01,2481.0,15562.0,0,0
2,0.0,0.00,3515.0,23643.0,0,0
3,10.0,0.00,0.0,0.0,1,1
4,0.0,0.00,2803.0,17469.0,0,0


### Feature Notes

Here's a breakdown of each feature currently in our feature vector `X`:

*   **`search_volume`**
    *   **Meaning**: The estimated average monthly search volume for the given keyword. This indicates potential audience interest.
    *   **Missing handling**: Assumed to be populated during initial data ingestion. No explicit `NaN` handling applied here, as it's expected to be a complete numerical field.
    *   **Categorical**: Numeric (integer).
    *   **Available-when**: Available at the time of prediction, as it's a standard metric from keyword research tools.

*   **`competition`**
    *   **Meaning**: A score representing the difficulty of ranking for a specific keyword, typically ranging from 0 to 1. Higher values indicate more competition.
    *   **Missing handling**: Assumed to be populated during initial data ingestion. No explicit `NaN` handling applied here, as it's expected to be a complete numerical field.
    *   **Categorical**: Numeric (float).
    *   **Available-when**: Available at the time of prediction, also a standard metric from keyword research tools.

*   **`word_count`**
    *   **Meaning**: The number of words in the content associated with the keyword.
    *   **Missing handling**: Missing values were filled with `0`. A binary flag `word_count_is_missing` was created to indicate original missingness.
    *   **Categorical**: Numeric (integer).
    *   **Available-when**: Available at the time of prediction, as it can be calculated directly from the content itself.

*   **`char_count`**
    *   **Meaning**: The number of characters in the content associated with the keyword.
    *   **Missing handling**: Missing values were filled with `0`. A binary flag `char_count_is_missing` was created to indicate original missingness.
    *   **Categorical**: Numeric (integer).
    *   **Available-when**: Available at the time of prediction, as it can be calculated directly from the content itself.

*   **`word_count_is_missing`**
    *   **Meaning**: A binary indicator (1 if `word_count` was originally missing, 0 otherwise). This captures information about the completeness of the content.
    *   **Missing handling**: Directly derived from missingness, so no further missing value handling is needed for this feature.
    *   **Categorical**: Binary (0/1).
    *   **Available-when**: Available at the time of prediction, as it's determined by the presence or absence of content.

*   **`char_count_is_missing`**
    *   **Meaning**: A binary indicator (1 if `char_count` was originally missing, 0 otherwise). Similar to `word_count_is_missing`, it signifies content completeness.
    *   **Missing handling**: Directly derived from missingness, so no further missing value handling is needed for this feature.
    *   **Categorical**: Binary (0/1).
    *   **Available-when**: Available at the time of prediction, as it's determined by the presence or absence of content.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### The Leakage Hunt: Identifying and Testing for Leakage

Data leakage occurs when information from the target variable, or information that would not be available at prediction time, is inadvertently used as a feature in the model. This can lead to overly optimistic performance estimates during training and validation.

In this section, we will examine several columns from the original `df` DataFrame that are highly suspicious of containing leaked information. These columns typically represent outcomes or performance metrics that a model would ideally *predict*, not *use as input*.

In [24]:
# Identify highly suspicious columns for leakage
leaky_candidates = [
    'google_organic_clicks',
    'google_organic_impressions',
    'pageviews',
    'sessions',
    'conversions',
    'revenue',
    'delta_rank_trend' # This often implies future knowledge about rank changes
]

# Check if these columns exist in the original DataFrame df
existing_leaky_candidates = [col for col in leaky_candidates if col in df.columns]

if existing_leaky_candidates:
    print("Potential Leaky Features Identified:")
    for col in existing_leaky_candidates:
        print(f"- {col}")

    # Calculate the correlation matrix for these potential leaky features
    # and display it to show their strong inter-relationships
    correlation_matrix = df[existing_leaky_candidates].corr()
    print("\nCorrelation Matrix of Potential Leaky Features:")
    display(correlation_matrix)

    print("\nObservation: A high correlation among these features indicates that they are likely measuring similar aspects of content performance or outcome. Including any of these directly as features would constitute severe data leakage, as they represent information that would not be available at the time a prediction is made. For example, if we were trying to predict 'google_organic_clicks', using 'pageviews' or 'revenue' as a feature would be leakage as these are also outcomes or directly related to the outcome.")
else:
    print("No common leaky candidate columns found in the DataFrame. This might mean the dataset was pre-processed to remove them, or different leakage sources should be considered.")

No common leaky candidate columns found in the DataFrame. This might mean the dataset was pre-processed to remove them, or different leakage sources should be considered.


### What was excluded and why

Based on our initial check, the `df` DataFrame did not contain the highly suspicious leaky candidate columns (`google_organic_clicks`, `google_organic_impressions`, `pageviews`, `sessions`, `conversions`, `revenue`, `delta_rank_trend`). This implies that the dataset provided has either been pre-processed to remove these outcome-related features, or they were never part of this specific `df`.

Therefore, we did not have to explicitly exclude any features based on the leakage hunt. However, it's crucial to state that by constructing our feature vector `X` with only:

*   `search_volume`
*   `competition`
*   `word_count`
*   `char_count`
*   `word_count_is_missing`
*   `char_count_is_missing`

**All other columns originally present in the `df` DataFrame have been implicitly excluded.** This exclusion is based on the principle that only features available at prediction time and not directly derived from the target should be included. Any columns not explicitly added to `X` are considered irrelevant or potentially leaky for our current model building purpose.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.